# <img style="float: left; padding-right: 15px; width: 35px" src="https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/iacs.png"> CS 1090B: Advanced Topics in Data Science 

# Homework 3: Dimensionality Reduction, Clustering, and Transfer Learning



**Harvard University**<br/>
**Spring 2026**<br/>
**Instructors**: Pavlos Protopapas, Kevin Rader, Chris Gumb


<hr style="height:2pt">

<!-- HW3-DISCLAIMER-CLUSTER -->
<div style="background-color:#fff3cd; color:#222222; border:1px solid #ffeeba; padding:12px 14px; border-radius:10px; margin:10px 0;">
<strong>Run environment:</strong> This homework is designed to be run on the <strong>academic cluster</strong>.
All required datasets are available on the cluster at: <code>~/163602/datasets/</code>
</div>


<!-- HW3-DISCLAIMER-CHECKPOINTING -->
<div style="background-color:#d1ecf1; color:#222222; border:1px solid #bee5eb; padding:12px 14px; border-radius:10px; margin:10px 0;">
<strong>Reproducibility / efficiency expectation:</strong> You are expected to write code that <strong>saves</strong> trained model weights (and any other expensive artifacts)
to disk and <strong>loads</strong> them on reruns to avoid unnecessary retraining. Your notebook should run quickly after the first successful training run.
</div>


## Notebook Contents
<a id = "contents"></a>

- [**Part I: Dimensionality Reduction & Clustering**](#part1):
  Wildlife Species Analysis with PCA, UMAP, and $k$-Means
  
- [**Part II: Transfer Learning**](#part2):
  CNN Classification of 10 Animal Classes
  
- [**Part III: Variational Autoencoders**](#part3):
  Generative Modeling & Image Synthesis

In [ ]:
# Standard library
import os
import gc
import time
import warnings

# Numerical computing and data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from PIL import Image

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights

# UMAP
import umap
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

# K-means reproducibility: number of random initializations per run
KMEANS_N_INIT = 100

from pathlib import Path
# Shared datasets on the course cluster
DATA_ROOT = Path('~/163602/datasets').expanduser()


In [ ]:
# measure notebook runtime
time_start = time.time()

<a id="part1"></a>

## <div class='exercise'>Part I: Wildlife Species Analysis with PCA, UMAP, and $k$-Means [40 Points]</div> 
In this section, you will apply dimensionality reduction techniques (PCA and UMAP) and clustering algorithms ($k$-means) to a dataset of wildlife images. Through visualization and experimentation with different feature representations and hyperparameters, you'll discover what enables unsupervised methods to uncover meaningful structure in visual data.

### Q1.1 - Data Loading & Exploration [3 Points]

The folder `wild_species` contains a collection of wildlife face images.

Load the images from this directory and create an 8×8 grid displaying a random sample of images from the dataset. **Set the random seed to 1090 for reproducibility.** After visualizing the data, report the total number of images in the dataset and their dimensions (height, width, channels).

**Deliverables:**

1. An 8×8 grid visualization of randomly sampled images (seed = 1090)
2. Print statements reporting:
   - Total number of images in the dataset
   - Image dimensions (height × width × channels)
3. A brief comment (2-3 sentences) describing what types of animals you observe in the sample and any notable visual characteristics or patterns you notice across the images.

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.2 - PCA Dimensionality Reduction on Raw Pixels [3 Points]

Now let's attempt to reduce the high-dimensional image data to 2 dimensions using Principal Component Analysis (PCA). 

Flatten each image into a vector, apply PCA to reduce the dimensionality to 2 components, and create a scatter plot of the results. Report the explained variance ratio for the first two principal components.

**Deliverables:**

1. A 2D scatter plot showing all images in PCA space
2. Print the explained variance ratio for PC1 and PC2, and their cumulative sum

**Hint:** Remember to standardize your features before applying PCA.

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.3 - Nearest Neighbors in PCA Space [4 Points]

To evaluate whether PCA is capturing meaningful structure, let's examine the nearest neighbors of randomly selected images in the reduced 2D space.

Randomly sample 3 images from the dataset (use seed 1090). For each sampled image, find its 23 nearest neighbors in the PCA-reduced space using Euclidean distance. For each of the 3 sampled images, create a 3×8 grid showing the original image (top-left) followed by its 23 nearest neighbors in order of increasing distance.

**Deliverables:**

1. Three separate 3×8 grids (one for each sampled image), where the first image is the query and the remaining 23 are its nearest neighbors in PCA space
2. A brief comment (2-3 sentences) on what you observe: are the nearest neighbors visually similar to the query image? What does this suggest about PCA's effectiveness on raw pixel values?

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.4 - UMAP on Raw Pixels [4 Points]

Now let's try a different dimensionality reduction technique: UMAP (Uniform Manifold Approximation and Projection). UMAP is often better at preserving local structure compared to PCA (or at least, we can explictly control for global vs. local focus).

Apply UMAP to the same raw pixel data with `n_neighbors = 50` and `min_dist = 0.1`, reducing to 2 dimensions. Then repeat the nearest neighbor visualization from Q1.3: for the same 3 sampled images (seed 1090), find their 23 nearest neighbors in the UMAP-reduced space and display them in 3×8 grids.

**Deliverables:**

1. A 2D scatter plot showing all images in UMAP space
2. Three separate 3×8 grids showing each query image and its 23 nearest neighbors in UMAP space
3. A brief comment (2-3 sentences) comparing the results to PCA: does UMAP do better? Are the neighbors more similar? If both methods struggle, what might be the underlying issue - is it the algorithms themselves, or could something else be going wrong with the features?

**Note:** Remember to set `random_state = 1090` for UMAP reproducibility, and ensure your 3 query images are the same ones you sampled in Q1.3.

In [ ]:
# 2D Scatter Plot of UMAP space
# your code here


In [ ]:
# Grids of Example Query Images and Their Nearest Neighbors
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.5 - Transfer Learning with ResNet50 [8 Points]

Raw pixel values aside, let's try a different approach: using a pre-trained deep neural network to extract meaningful features.

Read about [ResNet50](https://pytorch.org/vision/main/models/generated/torchvision.models.resnet50.html) in the PyTorch documentation. Use a pre-trained ResNet50 model (trained on ImageNet) with **IMAGENET1K_V1** weights to extract features from your images. Remove the final classification layer to get the feature representations from the second-to-last layer. Then apply PCA to reduce these ResNet features to 2 dimensions and visualize the results.

**Important:** All ImageNet pre-trained models require specific preprocessing. Your images must be:
- Resize the **shorter side** to 256 (preserving aspect ratio), then center-crop to 224×224
- Converted to tensors with pixel values in [0, 1]
- Normalized using ImageNet statistics: `mean = [0.485, 0.456, 0.406]`, `std = [0.229, 0.224, 0.225]`

**Deliverables:**

1. A 2D scatter plot showing all images in PCA space using ResNet50 features
2. Print the explained variance ratio for PC1 and PC2
3. A brief comment (2-3 sentences) on what you observe: does the visualization look different from Q1.2? Do you see any structure or clustering? What does this suggest about the quality of ResNet50 features compared to raw pixels?

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.6 - UMAP Hyperparameter Selection [4 Points]

Now let's apply UMAP to the ResNet50 features. Before doing so, read this excellent interactive article on [Understanding UMAP](https://pair-code.github.io/understanding-umap/) to build intuition about how UMAP works and how its hyperparameters affect the embedding.

Based on your understanding from the article and the nature of your dataset, choose an appropriate value for `n_neighbors`. Set `min_dist = 0.99` (this spreads points out maximally, helping us see how the dataset is structured without points clumping together). Apply UMAP to the ResNet50 features and visualize the results.

**Deliverables:**

1. A 2D scatter plot showing all images in UMAP space using ResNet50 features
2. A brief explanation (2-3 sentences) of your `n_neighbors` choice: what value did you select and why is it appropriate for this dataset? What aspect of the data structure (local vs global) are you trying to capture?
3. A comparison (2-3 sentences) between the UMAP and PCA results: do they show similar or different structure? What are the key differences you observe?

**Note:** Remember to set `random_state = 1090` for UMAP reproducibility.

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.7 - Systematic Grid Search: UMAP + $k$-Means [10 Points]

Now for the systematic exploration. We'll perform a comprehensive grid search over UMAP hyperparameters and $k$-means clustering to find the configuration that produces the best-defined clusters according to the silhouette score metric.

Build a pipeline that:
1. Loops over UMAP hyperparameters:
   - `n_neighbors`: [15, 50, 100, 150]
   - `min_dist`: [0.1, 0.2, 0.5, 0.99]
   - `n_components` (UMAP dimensions): [2, 15, 50]
   - `random_state`: 1090 (for all UMAP fits)
2. For each UMAP embedding, runs $k$-means with `k` ∈ [5, 6, 7, 8, 9, 10], `KMEANS_N_INIT` initializations per run, and `random_state = 1090`
3. Calculates the silhouette score for each clustering
4. Stores all results in a dataframe

**Deliverables:**

1. A dataframe containing all results with columns: `n_neighbors`, `min_dist`, `n_components`, `k`, `silhouette_score`
2. Display the top 15 rows sorted by silhouette score (highest first)
3. A brief analysis (3-4 sentences): what patterns do you observe in the best-performing configurations? In particular, comment on the role of `min_dist` - does it behave like a true hyperparameter (where the optimal value is uncertain) or does it consistently favor one end of the range, and why does this make sense given how $k$-means works? Does the optimal $k$ match your expectations based on the number of species you've seen in the images so far? What does this tell you about the structure in the data?

**Note:** This will take a few minutes to run (288 total combinations). Use `random_state = 1090` for both UMAP and $k$-means to ensure reproducible results.

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q1.8 - Visualizing and Evaluating the Best Clustering [4 Points]

Now let's visualize the results of our best clustering configuration from Q1.7.

From your Q1.7 results, select the best-performing configuration **among those with `n_components = 2`**. Fit UMAP with those hyperparameters and apply $k$-means to obtain cluster assignments. Then visualize the cluster assignments on this **2D** UMAP embedding (color points by cluster label). Finally, for each cluster, randomly sample and display 16 images.

**Deliverables:**

1. A 2D scatter plot using the UMAP embedding from your selected best **Q1.7** (with `n_components=2`) configuration, with points colored by cluster assignment (include a legend)
2. For each cluster, a 2×8 grid showing 16 randomly sampled images from that cluster (use seed 1090)
3. A brief evaluation (2-3 sentences): how visually coherent is each cluster? Do the images within each cluster look similar to each other? Do the clusters correspond to distinct species, or do you see mixing? What does this tell you about the quality of the unsupervised clustering?

**Note:** Use `random_state = 1090` for both UMAP and $k$-means to ensure reproducible cluster assignments, and `KMEANS_N_INIT` $k$-means initializations. Use `seed = 1090` when sampling images from each cluster.

In [ ]:
# Cluster Plot
# your code here


In [ ]:
# Examples of Images from Each Cluster
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


<a id="part2"></a>

## Part II: Image Classification & Transfer Learning [30 Points]

In this section, you will build and compare three approaches to image classification: a CNN trained from scratch, transfer learning with frozen pre-trained features, and fine-tuned transfer learning. By experimenting on a 10-class animal dataset, you'll discover how pre-trained models dramatically improve performance and understand the tradeoffs between training from scratch versus leveraging learned representations.

### Q2.1 - Data Loading & Visualization [3 Points]

The folder `animal_groups` contains a collection of animal images organized into 10 classes: butterfly, cat, chicken, cow, dog, elephant, horse, sheep, spider, and squirrel.

Load the training dataset and create a 10×7 grid displaying 7 randomly sampled images from each class. **Set the random seed to 1090 for reproducibility.** After visualizing the data, report the total number of images in each split (train/val/test) and the number of images per class in the training set.

**Deliverables:**

1. A 10×7 grid visualization where each row shows 7 randomly sampled training images from one class (seed = 1090)
2. Print statements reporting:
   - Total number of images in train, val, and test sets
   - Number of training images per class

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q2.2 - Baseline CNN from Scratch [14 Points]

Design and train the following convolutional neural network from scratch to classify the 10 animal classes. We provide the data loading pipeline and training utilities as starter code - your task is to implement the architecture and training loop as specified.

**Starter Code Provided:**
- Data loaders for train/val/test splits (128×128 images, batch size 64)
- Training data augmentation: random horizontal flips, rotations (±10°), and color jitter
- `train_epoch()` and `validate()` training utility functions
- `plot_training_history()` plotting utility

**Architecture (Implement Exactly as Described):**

Build a CNN with 4 convolutional blocks followed by a fully connected classifier:
- **Block 1**: Conv2d(3 → 32, kernel 3×3, padding 1) → BatchNorm → ReLU → MaxPool2d(2)
- **Block 2**: Conv2d(32 → 64, kernel 3×3, padding 1) → BatchNorm → ReLU → MaxPool2d(2)
- **Block 3**: Conv2d(64 → 128, kernel 3×3, padding 1) → BatchNorm → ReLU → MaxPool2d(2)
- **Block 4**: Conv2d(128 → 256, kernel 3×3, padding 1) → BatchNorm → ReLU → MaxPool2d(2)
- **Classifier**: Flatten → Linear(256×8×8, 512) → ReLU → Dropout(0.5) → Linear(512, 10)

**Training Specifications:**
- Optimizer: Adam with Learning Rate 0.001
- 10 Epochs
- Save the best model based on validation accuracy to `best_cnn_scratch.pth`

**Deliverables:**

1. **Architecture Summary**: Print your model architecture and total parameter count.

2. **Training History Plots**: Use the provided `plot_training_history()` function to create two side-by-side plots showing (a) training and validation loss over epochs, and (b) training and validation accuracy over epochs.

3. **Model Performance**: Save your final performance metrics to three variables: `cnn_train_acc` (final training accuracy), `cnn_val_acc` (best validation accuracy), and `cnn_test_acc` (test accuracy using best model).

**Note:** If implemented correctly, you should expect a validation accuracy **≥ 45%** by the end of 10 epochs. Note that random chance for a 10-class problem is 10%, so even this modest accuracy reflects meaningful learning!

**Note:** The provided helper functions report accuracy in **percent (0–100)**.


In [ ]:
# STARTER CODE - DO NOT EDIT

# Set seeds for reproducibility
torch.manual_seed(1090)
np.random.seed(1090)

# Check device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using Device: {device}\n")

def plot_training_history(train_losses, val_losses, train_accs, val_accs, title = ''):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (11, 4))
    
    ax1.plot(train_losses, label = 'Train Loss', linewidth = 2)
    ax1.plot(val_losses, label = 'Validation Loss', linewidth = 2)
    ax1.set_xlabel('Epoch', fontsize = 12)
    ax1.set_ylabel('Loss', fontsize = 12)
    ax1.set_title('Training and Validation Loss' + (f' - {title}' if title else ''), fontsize = 14)
    ax1.legend()
    ax1.grid(True, alpha = 0.3)
    
    ax2.plot(train_accs, label = 'Train Accuracy', linewidth = 2)
    ax2.plot(val_accs, label = 'Validation Accuracy', linewidth = 2)
    ax2.set_xlabel('Epoch', fontsize = 12)
    ax2.set_ylabel('Accuracy (%)', fontsize = 12)
    ax2.set_title('Training and Validation Accuracy' + (f' - {title}' if title else ''), fontsize = 14)
    ax2.legend()
    ax2.grid(True, alpha = 0.3)
    
    plt.tight_layout()
    plt.show()

# Data loading
train_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.RandomCrop(128),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], 
                       std = [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], 
                       std = [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(DATA_ROOT / 'animal_groups' / 'train', transform = train_transform)
val_dataset = datasets.ImageFolder(DATA_ROOT / 'animal_groups' / 'val', transform = val_test_transform)
test_dataset = datasets.ImageFolder(DATA_ROOT / 'animal_groups' / 'test', transform = val_test_transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / total, 100. * correct / total

In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


### Q2.3 - Frozen Transfer Learning [5 Points]

Now let's explore transfer learning by using a pre-trained ResNet50 model as a feature extractor. Instead of training a CNN from scratch, you'll leverage features learned from ImageNet (a dataset of 1.2M images across 1,000 categories) and only train a new classifier on top.

**Starter Code Provided:**
- Data loaders for train/val/test splits with 224×224 images to match ResNet50's expected input size (resize the shorter side to 256, then center crop to 224)

**Your Task:**
- Load ResNet50 pre-trained on ImageNet (use `IMAGENET1K_V1` weights)
- Freeze all convolutional layers (set `requires_grad = False`)
- Replace the final fully connected layer with a new classifier for 10 classes
- Train **only** the new classifier layer for exactly 5 epochs

**Deliverables:**

1. **Training History Plots**: Use the provided `plot_training_history()` function to create two side-by-side plots showing (a) training and validation loss over epochs, and (b) training and validation accuracy over epochs.

2. **Model Performance**: Save your final performance metrics to three variables: `frozen_train_acc` (final training accuracy), `frozen_val_acc` (best validation accuracy), and `frozen_test_acc` (test accuracy using best model).

In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q2.4 - Fine-Tuned Transfer Learning [5 Points]

Now let's go one step further: instead of keeping all ResNet50 layers frozen, we'll **unfreeze the last bottleneck block** and fine-tune it along with the classifier. This allows the model to adapt the high-level features specifically to our animal dataset.

**Approach:**
- Load ResNet50 pre-trained on ImageNet (use `IMAGENET1K_V1` weights)
- Freeze all layers initially
- Replace the final fully connected layer with a new classifier for 10 classes
- **Unfreeze only the last bottleneck block** (`layer4[2]`) and the final classifier
- Train for exactly 10 epochs with differential learning rates (lower LR for pre-trained layers, higher for new classifier)

**Important:** Reuse the 224×224 data loaders from Q2.3 (`train_loader`, `val_loader`, `test_loader`).

**Deliverables:**

1. **Training History Plots**: Create two side-by-side plots showing (a) training and validation loss over epochs, and (b) training and validation accuracy over epochs.

2. **Model Performance**: Save your final performance metrics to three variables: `finetuned_train_acc` (final training accuracy), `finetuned_val_acc` (best validation accuracy), and `finetuned_test_acc` (test accuracy using best model).

**Hint:** Use differential learning rates in your optimizer - a lower learning rate (e.g., 0.0001) for the pre-trained layers being fine-tuned, and a higher rate (e.g., 0.001) for the new classifier head.

In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q2.5 - Comparison & Analysis [3 Points]

Now that you've trained three different models, let's compare their performance and analyze the results.

**Deliverables:**

1. **Comparison Table**: Create a pandas DataFrame with three rows (one per approach: CNN from Scratch, Frozen Transfer Learning, Fine-Tuned Transfer Learning) and three columns (Train Accuracy, Validation Accuracy, Test Accuracy). Display the table with accuracies formatted to 2 decimal places.

2. **Discussion** (1 paragraph): Address the following questions in your analysis:
   - Which approach performed best overall? By how much did it improve over the baseline CNN?
   - How do the frozen and fine-tuned transfer learning results compare? Was fine-tuning worth the additional trainable parameters?
   - Looking at train vs. validation/test accuracy gaps, which model shows signs of overfitting? Why might this be?
   - Based on these results, when would you recommend training from scratch vs. using transfer learning?

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


<a id="part3"></a>

## Part III: Variational Autoencoders [30 Points]
In this section, you will explore generative modeling using autoencoders. Starting with a vanilla convolutional autoencoder and progressing to a conditional variational autoencoder (CVAE), you will learn how to compress images into meaningful latent representations, reconstruct them, and ultimately condition the generation process on class labels to perform cross-class image synthesis.

### Q3.1 - Data Loading & Exploration [4 Points]

In this final part of the homework, we shift our focus to generative modeling. We'll be working with the `humans_pets` dataset containing face images of humans, cats, and dogs.

Load the training data from the `humans_pets` directory and create a 3×7 grid displaying 7 randomly sampled images from each class (seed = 1090). Report the total number of images in each split and the number of images per class in the training set.

**Deliverables:**

1. A 3×7 grid visualization of randomly sampled training images, one row per class (seed = 1090)
2. Print statements reporting:
   - Total number of images in train, val, and test sets
   - Number of training images per class

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q3.2 - Convolutional Autoencoder [10 Points]

Now let's build and train a convolutional autoencoder on the `humans_pets` dataset. An autoencoder learns to compress images into a compact latent representation and then reconstruct them, forcing the model to capture the most important visual features in a low-dimensional space.

We provide the data loaders and model architecture as starter code - your task is to implement the training loop and visualizations.

**Starter Code Provided:**
- Data loaders for train and val splits (128×128 images)
- `ConvAutoencoder` architecture: 4 convolutional blocks encoder → 128-dimensional latent space → 4 transposed convolutional blocks decoder

**Your Task:**
- Implement the training loop for exactly 10 epochs using MSE reconstruction loss and Adam optimizer (lr = 0.001)
- Track and plot both training and validation reconstruction loss
- Visualize original vs. reconstructed images for each class

**Deliverables:**

1. **Loss Plot**: A single plot showing training and validation reconstruction loss over 10 epochs.

2. **Reconstruction Grid**: A 6×6 grid showing original (odd rows) and reconstructed (even rows) images, with one pair of rows per class.

3. **Discussion** (2-3 sentences): How well does the autoencoder reconstruct each class? Which class reconstructs best and which worst? Why might this be? What does this tell you about the complexity of each class's visual features?

**Note:** Use `random_state = 1090` for reproducibility. Training should take approximately 5-10 minutes.

In [ ]:
# Helper Code - DO NOT DELETE
# Data loading
ae_transform = transforms.Compose([
    transforms.Resize(128),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5, 0.5, 0.5], 
                        std = [0.5, 0.5, 0.5])
])

ae_train_dataset = datasets.ImageFolder(DATA_ROOT / 'humans_pets' / 'train', transform = ae_transform)
ae_val_dataset = datasets.ImageFolder(DATA_ROOT / 'humans_pets' / 'val', transform = ae_transform)

ae_train_loader = DataLoader(ae_train_dataset, batch_size = 64, shuffle = True, num_workers = 2)
ae_val_loader = DataLoader(ae_val_dataset, batch_size = 64, shuffle = False, num_workers = 2)

# Autoencoder Architecture
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim = 128):
        super(ConvAutoencoder, self).__init__()
        
        # Encoder: 128x128x3 -> latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 4, stride = 2, padding = 1),    # 64x64
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size = 4, stride = 2, padding = 1),   # 32x32
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size = 4, stride = 2, padding = 1),  # 16x16
            nn.ReLU(),
            nn.Conv2d(128, 256, kernel_size = 4, stride = 2, padding = 1), # 8x8
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, latent_dim)
        )
        
        # Decoder: latent_dim -> 128x128x3
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (256, 8, 8)),
            nn.ConvTranspose2d(256, 128, kernel_size = 4, stride = 2, padding = 1), # 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size = 4, stride = 2, padding = 1),  # 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size = 4, stride = 2, padding = 1),   # 64x64
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size = 4, stride = 2, padding = 1),    # 128x128
            nn.Tanh()
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

# Initialize model
ae_model = ConvAutoencoder(latent_dim = 128).to(device)
print(f"Total Parameters: {sum(p.numel() for p in ae_model.parameters()):,}")

def denormalize(tensor):
    return (tensor * 0.5 + 0.5).clamp(0, 1)

In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


### Q3.3 - Conditional Variational Autoencoder [16 Points]

Conditional VAEs are similar to standard VAEs, except they incorporate a class label directly into the latent space. When trained this way, the model learns to distinguish between features associated with each class, allowing you to explicitly "activate" class attributes in the latent space. This means you can take an image of one class and ask the decoder to reconstruct it as a different class entirely - for example, encoding a human face and decoding it as a cat or dog.

**Architecture (Implement Exactly as Described):**

**Encoder**:
  - Conv2d(3 → 32, kernel 4×4, stride 2) → ReLU → 64×64
  - Conv2d(32 → 64, kernel 4×4, stride 2) → ReLU → 32×32
  - Conv2d(64 → 128, kernel 4×4, stride 2) → ReLU → 16×16
  - Conv2d(128 → 256, kernel 4×4, stride 2) → ReLU → 8×8
  - Flatten → concatenate with one-hot class label (3 dimensions)
  - Linear(256×8×8 + 3, 128) → $\mu$
  - Linear(256×8×8 + 3, 128) → $\log\sigma^2$
  - Reparameterization: $z = \mu + \epsilon \cdot \sigma$, $\epsilon \sim \mathcal{N}(0, I)$

**Decoder**:
  - Concatenate $z$ with one-hot class label
  - Linear(128 + 3, 256×8×8) → ReLU → Unflatten to (256, 8, 8)
  - ConvTranspose2d(256 → 128, kernel 4×4, stride 2) → ReLU → 16×16
  - ConvTranspose2d(128 → 64, kernel 4×4, stride 2) → ReLU → 32×32
  - ConvTranspose2d(64 → 32, kernel 4×4, stride 2) → ReLU → 64×64
  - ConvTranspose2d(32 → 3, kernel 4×4, stride 2) → Tanh → 128×128

**Loss**: MSE reconstruction loss + $\beta \cdot$ KL divergence, where $\beta = 0.2$:
$$\mathcal{L} = \text{MSE}(x, \hat{x}) + \beta \cdot \left( -\frac{1}{2} \sum_{j} \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2 \right) \right)$$

**Training Specifications:**
- Optimizer: Adam with learning rate 0.001
- Exactly 10 epochs
- Gradient clipping with `max_norm = 1.0`
- Save best model (lowest validation loss) to `best_cvae.pth`
- Reuse data loaders from Q3.2 (`ae_train_loader`, `ae_val_loader`)

**Deliverables:**

1. **Loss Plot**: A single plot showing training and validation loss over 10 epochs.

2. **Cross-Class Reconstruction Grid**: A 9×6 grid (6 image samples per row) showing for each class: the original images, reconstructions as each of the other two classes. For example, for cats: original cats → reconstructed as dogs → reconstructed as humans.

3. **Discussion** (3-4 sentences): How well does the CVAE perform cross-class reconstruction? Can you see class-specific features being applied (e.g., fur texture on humans, human-like features on cats)? What are the limitations of this approach?

**Note:** Training should take approximately 5-10 minutes. Use `random_state = 1090` for reproducibility.

In [ ]:
# CVAE Architecture
# your code here


In [ ]:
# your code here


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*


<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


## Wrap-Up

In a few sentences, describe the aspect(s) of the assignment you found most challenging. This could be conceptual (e.g., understanding the VAE loss, interpreting clustering results) and/or related to coding and implementation (e.g., debugging the training loop, getting the architecture right).

Additionally, store the number of hours you spent working on this assignment as an integer or float in the variable `time_spent_on_hw`.

In [ ]:
time_spent_on_hw = ...


<!-- ANSWER-BOUNDARY-START -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


*Your Answer Here...*



<!-- ANSWER-BOUNDARY-END -->
<div style="background-color:#f0f7ff; border:1px solid #cfe6ff; height:14px; border-radius:8px; margin:10px 0;"></div>


In [ ]:
assert float(time_spent_on_hw),\
    "Please select a time in hours (int or float) to specify how long you spent on this assignment."

In [ ]:
time_end = time.time()
print(f"It took {(time_end - time_start)/60:.2f} minutes for this notebook to run!")

**This concludes HW3. Thank you!** 🌈